In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.pipeline import Pipeline
from sklearn.metrics import (classification_report, accuracy_score, precision_score, recall_score, f1_score)

In [2]:
spam_dataset = pd.read_csv('spam.csv', encoding="ISO-8859-1", usecols=[0, 1], names=['Spam', 'Text'],skiprows=1)

spam_dataset['Spam'] = spam_dataset['Spam'].replace(['ham', 'spam'], [0, 1])

spam_dataset

C:\Users\atrel\AppData\Local\Temp\ipykernel_6112\934167254.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  spam_dataset['Spam'] = spam_dataset['Spam'].replace(['ham', 'spam'], [0, 1])


,Spam,Text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,1,This is the 2nd time we have tried 2 contact u...
5568,0,Will Ì_ b going to esplanade fr home?
5569,0,"Pity, * was in mood for that. So...any other s..."
5570,0,The guy did some bitching but I acted like i'd...


In [3]:
X = spam_dataset['Text'].astype(str)
y = spam_dataset['Spam'].astype('int64').to_numpy().ravel()

print(np.unique(y))
print(y.dtype)

[0 1]
int64


In [4]:
# podzial na zbior treningowy i testowy ze stratyfikacja

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=0,
    stratify=y)

In [5]:
# Pierwszy RandomForest w Pipeline

first_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(2, 2),
        max_df=0.5,
        min_df=0.001,
        use_idf=True)),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=0,
        n_jobs=-1))])

first_pipeline.fit(X_train, y_train)

print("Score pierwszego modelu:")
print(first_pipeline.score(X_test, y_test))

Score pierwszego modelu:
0.9605381165919282


In [6]:
# Feature importance - cechy z modelu z Pipeline

feature_importances = first_pipeline.named_steps['model'].feature_importances_
feature_names = np.array(first_pipeline.named_steps['tfidf'].get_feature_names_out())

feature_importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importances}).sort_values(by='importance', ascending=False)

print(feature_importance_df.head(20))

              feature  importance
273             co uk    0.035779
1603      your mobile    0.026403
1266         to claim    0.020486
1126          stop to    0.019502
501          have won    0.017661
998       please call    0.015648
1542         you have    0.012939
1085           sms ac    0.010671
1061        send stop    0.010581
953           opt out    0.010067
1008           po box    0.009894
1251          to 8007    0.009803
1344         txt stop    0.009422
1269       to contact    0.008641
1294            to no    0.007777
221   call mobileupd8    0.007730
394          for your    0.007381
1267       to collect    0.007347
1212         the word    0.007087
1037       reply with    0.006978


In [7]:
# Importance > 0.001

selected_features = feature_importance_df[
    feature_importance_df['importance'] > 0.001]['feature'].values

print("Liczba wybranych cech wedlug pierwszego modelu:")
print(len(selected_features))

Liczba wybranych cech wedlug pierwszego modelu:
244


In [8]:
# Pipeline: TF-IDF/selekja/RF+GridSearch

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(2, 2),
        max_df=0.5,
        min_df=0.001,
        use_idf=True)),
    ('selector', SelectFromModel(
        estimator=RandomForestClassifier(
            n_estimators=200,
            random_state=0,
            n_jobs=-1),
        threshold=0.001)),
    ('model', RandomForestClassifier(
        random_state=0,
        n_jobs=-1))])

# mniejsza siatka: 2 x 2 x 2 =8 kombinacji
# przy cv=5 daje to 40 modeli zamiast 1080

param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 20],
    'model__min_samples_leaf': [1, 2]}

grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1)

grid.fit(X_train, y_train)

print("Najlepsze parametry:")
print(grid.best_params_)

print("Najlepszy wynik CV:")
print(grid.best_score_)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
Najlepsze parametry:
{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__n_estimators': 100}
Najlepszy wynik CV:
0.8394004327233004


In [9]:
# sprawdzenie wybranych cech z najlepszego Pipeline

best_model = grid.best_estimator_

best_tfidf = best_model.named_steps['tfidf']
best_selector = best_model.named_steps['selector']

best_feature_names = np.array(best_tfidf.get_feature_names_out())
selected_mask = best_selector.get_support()

selected_features_grid = best_feature_names[selected_mask]
selected_importances_grid = best_selector.estimator_.feature_importances_[selected_mask]

selected_features_df = pd.DataFrame({
    'feature': selected_features_grid,
    'importance': selected_importances_grid}).sort_values(by='importance', ascending=False)

print("Liczba wybranych cech w najlepszym Pipeline:")
print(len(selected_features_grid))

print(selected_features_df.head(20))

Liczba wybranych cech w najlepszym Pipeline:
244
             feature  importance
56             co uk    0.035779
241      your mobile    0.026403
194         to claim    0.020486
166          stop to    0.019502
91          have won    0.017661
142      please call    0.015648
235         you have    0.012939
162           sms ac    0.010671
158        send stop    0.010581
131          opt out    0.010067
144           po box    0.009894
188          to 8007    0.009803
211         txt stop    0.009422
196       to contact    0.008641
198            to no    0.007777
39   call mobileupd8    0.007730
77          for your    0.007381
195       to collect    0.007347
182         the word    0.007087
153       reply with    0.006978


In [10]:
# Sprawdzenie na zb. testowym, do predykcji przekazujemy surowy tekst, bo Pipeline sam robi TF-IDF i selekcje cech

y_pred_grid = best_model.predict(X_test)

print("Accuracy na zbiorze testowym:")
print(accuracy_score(y_test, y_pred_grid))

print("Classification report:")
print(classification_report(y_test, y_pred_grid, zero_division=0))

Accuracy na zbiorze testowym:
0.9614349775784753
Classification report:
              precision    recall  f1-score   support

           0       0.96      0.99      0.98       966
           1       0.95      0.75      0.84       149

    accuracy                           0.96      1115
   macro avg       0.96      0.87      0.91      1115
weighted avg       0.96      0.96      0.96      1115



In [11]:
# CountVectonizer w pipeline

count_pipeline = Pipeline([
    ('count', CountVectorizer(
        min_df=0.001,
        max_df=0.5,
        ngram_range=(2, 2))),
    ('model', RandomForestClassifier(
        n_estimators=200,
        random_state=0,
        n_jobs=-1))])

count_pipeline.fit(X_train, y_train)

print("Score dla CountVectorizer:")
print(count_pipeline.score(X_test, y_test))

print("Przykladowe elementy Vocabulary CountVectorizer:")
print(dict(list(count_pipeline.named_steps['count'].vocabulary_.items())[:20]))

Score dla CountVectorizer:
0.9542600896860987
Przykladowe elementy Vocabulary CountVectorizer:
{'but have': 190, 'have to': 499, 'to pay': 1297, 'got it': 458, 'wan me': 1399, 'now or': 895, 'co uk': 273, 'or call': 957, 'you must': 1556, 'lt gt': 739, 'am in': 73, 'had your': 479, 'your mobile': 1603, 'update to': 1360, 'to the': 1313, 'the latest': 1185, 'latest camera': 687, 'for free': 380, 'text yes': 1149, 'watching tv': 1413}


In [12]:
# Porownanie modeli

# Model TF-IDF przed selekcja cech
y_pred_tfidf = first_pipeline.predict(X_test)

wynik_tfidf = accuracy_score(y_test, y_pred_tfidf)
precision_tfidf = precision_score(y_test, y_pred_tfidf, zero_division=0)
recall_tfidf = recall_score(y_test, y_pred_tfidf, zero_division=0)
f1_tfidf = f1_score(y_test, y_pred_tfidf, zero_division=0)


# Model TF-IDF po selekcji cech i GridSearch
y_pred_tfidf_grid = best_model.predict(X_test)

wynik_tfidf_grid = accuracy_score(y_test, y_pred_tfidf_grid)
precision_tfidf_grid = precision_score(y_test, y_pred_tfidf_grid, zero_division=0)
recall_tfidf_grid = recall_score(y_test, y_pred_tfidf_grid, zero_division=0)
f1_tfidf_grid = f1_score(y_test, y_pred_tfidf_grid, zero_division=0)


# Model CountVectorizer
y_pred_count = count_pipeline.predict(X_test)

wynik_count = accuracy_score(y_test, y_pred_count)
precision_count = precision_score(y_test, y_pred_count, zero_division=0)
recall_count = recall_score(y_test, y_pred_count, zero_division=0)
f1_count = f1_score(y_test, y_pred_count, zero_division=0)


# Tabela do porownania
porownanie = pd.DataFrame({
    'Model': [
        'Random Forest + TF-IDF',
        'Random Forest + TF-IDF + selekcja cech + Pipeline + GridSearch',
        'Random Forest + CountVectorizer'],

    'Accuracy': [
        wynik_tfidf,
        wynik_tfidf_grid,
        wynik_count],

    'Precision': [
        precision_tfidf,
        precision_tfidf_grid,
        precision_count],

    'Recall': [
        recall_tfidf,
        recall_tfidf_grid,
        recall_count],

    'F1-score': [
        f1_tfidf,
        f1_tfidf_grid,
        f1_count]})

porownanie.sort_values(by='F1-score', ascending=False)

,Model,Accuracy,Precision,Recall,F1-score
1,Random Forest + TF-IDF + selekcja cech + Pipel...,0.961435,0.949153,0.751678,0.838951
0,Random Forest + TF-IDF,0.960538,0.956522,0.738255,0.833333
2,Random Forest + CountVectorizer,0.954260,0.895161,0.744966,0.813187
